# Mixture prior estimation for MASH

This notebook computes a data-driven prior mixture (a set of covariance matrices) for a MASH model, independent of the downstream analysis method. It applies factor analysis (FLASH, PCA), canonical covariances, residual-variance (Vhat) estimation, and Empirical Bayes matrix methods (Extreme Deconvolution `ed_bovy`, Ultimate Deconvolution `ud`) to build the prior, then plots the resulting covariance patterns.

## Description

The pipeline proceeds in four stages: (1) factor analyses (`flash`, `flash_nonneg`, `pca`, `canonical`) to derive candidate covariance structures; (2) residual-variance estimation (`vhat_*` workflows) to estimate the error covariance Vhat; (3) multivariate mixture estimation (`ud`, `ud_unconstrained`, `ed_bovy`) to fit the data-driven prior; and (4) `plot_U` to visualise the estimated covariance matrices.

## Input

- `--data`: a formatted association summary-statistics RDS (a list with `strong.z`/`strong.b`/`strong.s`, `random.*`, `null.*`, and `XtX`), e.g. produced upstream by `extract_effects`.
- `--output-prefix`: prefix for output files (derived from `--data` if omitted).
- `--effect-model`: `EE` (exchangeable effects) or `EZ` (exchangeable z-scores).
- `--vhat`: residual-variance method (`identity`, `simple`, `mle`, `vhat_corshrink_xcondition`, `vhat_simple_specific`).

## Output

- Residual variance estimate, the MASH data-driven prior (`*.<effect_model>.prior.rds`), and prior covariance plots.

## Steps

**Step 1.** Perform FLASH factor analysis:

**Timing:** ~5-15 min on typical compute infrastructure.

In [ ]:
sos run pipeline/mixture_prior.ipynb flash \
    --output-prefix protocol_example \
    --data input/mash/protocol_example.mashr_input.rds \
    --cwd output/mixture_prior

**Step 2.** Perform FLASH analysis with a non-negative factor constraint:

In [ ]:
sos run pipeline/mixture_prior.ipynb flash_nonneg \
    --output-prefix protocol_example \
    --data input/mash/protocol_example.mashr_input.rds \
    --cwd output/mixture_prior

**Step 3.** Derive covariances from principal components:

In [ ]:
sos run pipeline/mixture_prior.ipynb pca \
    --output-prefix protocol_example \
    --data input/mash/protocol_example.mashr_input.rds \
    --cwd output/mixture_prior

**Step 4.** Build canonical (single-condition and shared) covariances:

In [ ]:
sos run pipeline/mixture_prior.ipynb canonical \
    --output-prefix protocol_example \
    --data input/mash/protocol_example.mashr_input.rds \
    --cwd output/mixture_prior

**Step 5.** Estimate residual variance Vhat using the "identity" method:

In [ ]:
sos run pipeline/mixture_prior.ipynb vhat_identity \
    --output-prefix protocol_example \
    --data input/mash/protocol_example.mashr_input.rds \
    --cwd output/mixture_prior

**Step 6.** Estimate residual variance Vhat using the "simple" method (null z-scores):

In [ ]:
sos run pipeline/mixture_prior.ipynb vhat_simple \
    --output-prefix protocol_example \
    --data input/mash/protocol_example.mashr_input.rds \
    --cwd output/mixture_prior

**Step 7.** Estimate residual variance Vhat using the "mle" method:

In [ ]:
sos run pipeline/mixture_prior.ipynb vhat_mle \
    --output-prefix protocol_example \
    --data input/mash/protocol_example.mashr_input.rds \
    --cwd output/mixture_prior

**Step 8.** Estimate Vhat per condition via corshrink:

In [ ]:
sos run pipeline/mixture_prior.ipynb vhat_corshrink_xcondition \
    --output-prefix protocol_example \
    --data input/mash/protocol_example.mashr_input.rds \
    --cwd output/mixture_prior

**Step 9.** Estimate Vhat per condition via the "simple" method:

In [ ]:
sos run pipeline/mixture_prior.ipynb vhat_simple_specific \
    --output-prefix protocol_example \
    --data input/mash/protocol_example.mashr_input.rds \
    --cwd output/mixture_prior

**Step 10.** Compute the multivariate mixture prior via Ultimate Deconvolution (udr):

In [ ]:
sos run pipeline/mixture_prior.ipynb ud \
    --output-prefix protocol_example \
    --data input/mash/protocol_example.mashr_input.rds \
    --cwd output/mixture_prior

**Step 11.** Compute the mixture prior via unconstrained Ultimate Deconvolution:

In [ ]:
sos run pipeline/mixture_prior.ipynb ud_unconstrained \
    --output-prefix protocol_example \
    --data input/mash/protocol_example.mashr_input.rds \
    --cwd output/mixture_prior

**Step 12.** Compute the mixture prior via Extreme Deconvolution (Bovy et al.):

In [ ]:
sos run pipeline/mixture_prior.ipynb ed_bovy \
    --output-prefix protocol_example \
    --data input/mash/protocol_example.mashr_input.rds \
    --cwd output/mixture_prior

**Step 13.** Plot the estimated prior covariance patterns:

In [ ]:
sos run pipeline/mixture_prior.ipynb plot_U \
    --output-prefix protocol_example_plots \
    --data output/mixture_prior/protocol_example.EE.prior.rds \
    --cwd output/mixture_prior

## Command interface

In [ ]:
sos run pipeline/mixture_prior.ipynb -h

## Workflow implementation

## Global parameters

In [2]:
[global]
parameter: cwd = path('./mashr_workflow_output')
# Input summary statistics data
parameter: data = path
# Prefix of output files. If not specified, it will derive it from data.
# If it is specified, for example, `--output-prefix AnalysisResults`
# It will save output files as `{cwd}/AnalysisResults*`.
parameter: output_prefix = ''
parameter: output_suffix = 'all'
# Exchangable effect (EE) or exchangable z-scores (EZ)
parameter: effect_model = 'EE'
# Identifier of $\hat{V}$ estimate file
# Options are "identity", "simple", "mle", "vhat_corshrink_xcondition", "vhat_simple_specific"
parameter: vhat = 'simple'
parameter: mixture_components = ['flash', 'flash_nonneg', 'pca',"canonical"]
parameter: container = ""
# For cluster jobs, number commands to run per job
parameter: job_size = 1
# Wall clock time expected
parameter: walltime = "5h"
# Memory expected
parameter: mem = "16G"
# Number of threads
parameter: numThreads = 1
data = data.absolute()
cwd = cwd.absolute()
if len(output_prefix) == 0:
    output_prefix = f"{data:bn}"
prior_data = file_target(f"{cwd:a}/{output_prefix}.{effect_model}.prior.rds")
vhat_data = file_target(f"{cwd:a}/{output_prefix}.{effect_model}.V_{vhat}.rds")

## Factor analyses

In [ ]:
[flash]
input: data
output: f"{cwd}/{output_prefix}.flash.rds"
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime,  mem = mem, tags = f'{step_name}_{_output:bn}'
bash: expand = "${ }", stderr = f'{_output:n}.stderr', stdout = f'{_output:n}.stdout', container = container
    Rscript code/script/pecotmr_integration/mash_covariance.R \
        --data ${_input} \
        --component flash \
        --effect-model ${effect_model} \
        --output ${_output}

In [ ]:
[flash_nonneg]
input: data
output: f"{cwd}/{output_prefix}.flash_nonneg.rds"
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime,  mem = mem, tags = f'{step_name}_{_output:bn}'
bash: expand = "${ }", stderr = f'{_output:n}.stderr', stdout = f'{_output:n}.stdout', container = container
    Rscript code/script/pecotmr_integration/mash_covariance.R \
        --data ${_input} \
        --component flash_nonneg \
        --effect-model ${effect_model} \
        --output ${_output}

In [ ]:
[pca]
# Number of components in PCA analysis for prior
# set to 3 as in mash paper
parameter: npc = 2
input: data
output: f"{cwd}/{output_prefix}.pca.rds"
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime,  mem = mem, tags = f'{step_name}_{_output:bn}'
bash: expand = "${ }", stderr = f'{_output:n}.stderr', stdout = f'{_output:n}.stdout', container = container
    Rscript code/script/pecotmr_integration/mash_covariance.R \
        --data ${_input} \
        --component pca \
        --npc ${npc} \
        --effect-model ${effect_model} \
        --output ${_output}

In [ ]:
[canonical]
input: data
output: f"{cwd}/{output_prefix}.canonical.rds"
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime,  mem = mem, tags = f'{step_name}_{_output:bn}'
bash: expand = "${ }", stderr = f'{_output:n}.stderr', stdout = f'{_output:n}.stdout', container = container
    Rscript code/script/pecotmr_integration/mash_covariance.R \
        --data ${_input} \
        --component canonical \
        --effect-model ${effect_model} \
        --output ${_output}

## Estimate residual variance


In [6]:
# V estimate: "identity" method
[vhat_identity]
input: data
output: f'{vhat_data:nn}.V_identity.rds'
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime,  mem = mem, tags = f'{step_name}_{_output:bn}'
bash: expand = "${ }", stderr = f"{_output:n}.stderr", stdout = f"{_output:n}.stdout", container = container
    Rscript code/script/pecotmr_integration/mash_vhat.R \
        --data ${_input} \
        --method identity \
        --effect-model ${effect_model} \
        --output ${_output}

In [7]:
# V estimate: "simple" method (null z-scores)
[vhat_simple]
input: data
output: f'{vhat_data:nn}.V_simple.rds'
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime,  mem = mem, tags = f'{step_name}_{_output:bn}'
bash: expand = "${ }", stderr = f"{_output:n}.stderr", stdout = f"{_output:n}.stdout", container = container
    Rscript code/script/pecotmr_integration/mash_vhat.R \
        --data ${_input} \
        --method simple \
        --effect-model ${effect_model} \
        --output ${_output}

In [8]:
# V estimate: "mle" method (mash_estimate_corr_em on a random subset; needs the prior U)
[vhat_mle]
# number of samples to use
parameter: n_subset = 6000
# maximum number of iterations
parameter: max_iter = 6
input: data, prior_data
output: f'{vhat_data:nn}.V_mle.rds'
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime,  mem = mem, tags = f'{step_name}_{_output:bn}'
bash: expand = "${ }", stderr = f"{_output:n}.stderr", stdout = f"{_output:n}.stdout", container = container
    Rscript code/script/pecotmr_integration/mash_vhat.R \
        --data ${_input[0]} \
        --method mle \
        --prior-data ${_input[1]} \
        --n-subset ${n_subset} \
        --max-iter ${max_iter} \
        --effect-model ${effect_model} \
        --output ${_output}

In [ ]:
# V estimate: "corshrink" adaptive-shrinkage null correlation (run globally on the null set; the legacy per-gene GTEx SumstatQuery path is not portable)
[vhat_corshrink_xcondition]
input: data
output: f'{vhat_data:nn}.V_vhat_corshrink_xcondition.rds'
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime,  mem = mem, tags = f'{step_name}_{_output:bn}'
bash: expand = "${ }", stderr = f"{_output:n}.stderr", stdout = f"{_output:n}.stdout", container = container
    Rscript code/script/pecotmr_integration/mash_vhat.R \
        --data ${_input} \
        --method corshrink \
        --effect-model ${effect_model} \
        --output ${_output}

In [ ]:
# V estimate: "simple_specific" nearPD(cov(nullZ)) (run globally on the null set; legacy per-gene GTEx path not portable)
[vhat_simple_specific]
input: data
output: f'{vhat_data:nn}.V_vhat_simple_specific.rds'
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime,  mem = mem, tags = f'{step_name}_{_output:bn}'
bash: expand = "${ }", stderr = f"{_output:n}.stderr", stdout = f"{_output:n}.stdout", container = container
    Rscript code/script/pecotmr_integration/mash_vhat.R \
        --data ${_input} \
        --method simple_specific \
        --effect-model ${effect_model} \
        --output ${_output}

## Compute multivariate mixture

In [ ]:
# Prior engine: udr ED update (opt-in; known numerical issues)
[ud]
input: [data, vhat_data if vhat != "mle" else f'{vhat_data:nn}.V_simple.rds'] + [f"{cwd}/{output_prefix}.{m}.rds" for m in mixture_components]
output: prior_data
comp_files = ",".join([f"{cwd}/{output_prefix}.{m}.rds" for m in mixture_components])
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime,  mem = mem, tags = f'{step_name}_{_output:bn}'
bash: expand = "${ }", stderr = f"{_output:n}.stderr", stdout = f"{_output:n}.stdout", container = container
    Rscript code/script/pecotmr_integration/mash_prior.R \
        --data ${_input[0]} \
        --engine ud \
        --vhat-data ${_input[1]} \
        --component-files ${comp_files} \
        --effect-model ${effect_model} \
        --output ${_output}

In [ ]:
# Prior engine: udr TED update (opt-in; needs i.i.d./z-scale data)
[ud_unconstrained]
input: [data, vhat_data if vhat != "mle" else f'{vhat_data:nn}.V_simple.rds'] + [f"{cwd}/{output_prefix}.{m}.rds" for m in mixture_components]
output: prior_data
comp_files = ",".join([f"{cwd}/{output_prefix}.{m}.rds" for m in mixture_components])
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime,  mem = mem, tags = f'{step_name}_{_output:bn}'
bash: expand = "${ }", stderr = f"{_output:n}.stderr", stdout = f"{_output:n}.stdout", container = container
    Rscript code/script/pecotmr_integration/mash_prior.R \
        --data ${_input[0]} \
        --engine ud_ted \
        --vhat-data ${_input[1]} \
        --component-files ${comp_files} \
        --effect-model ${effect_model} \
        --output ${_output}

In [ ]:
# Prior engine: mashr extreme deconvolution (the exported bovy ED)
[ed_bovy]
input: [data, vhat_data if vhat != "mle" else f'{vhat_data:nn}.V_simple.rds'] + [f"{cwd}/{output_prefix}.{m}.rds" for m in mixture_components]
output: prior_data
comp_files = ",".join([f"{cwd}/{output_prefix}.{m}.rds" for m in mixture_components])
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime,  mem = mem, tags = f'{step_name}_{_output:bn}'
bash: expand = "${ }", stderr = f"{_output:n}.stderr", stdout = f"{_output:n}.stdout", container = container
    Rscript code/script/pecotmr_integration/mash_prior.R \
        --data ${_input[0]} \
        --engine cov_ed \
        --vhat-data ${_input[1]} \
        --component-files ${comp_files} \
        --effect-model ${effect_model} \
        --output ${_output}

## Plot patterns of sharing

This is a simple utility function that takes the output from the pipeline above and make some heatmap to show major patterns of multivariate effects. The plots will be ordered by their mixture weights.

## Anticipated Results

The pipeline produces output files in the `output/` subdirectory named after the workflow step. Verify success by checking that output files exist and are non-empty. See the **Output** section above for the expected file names and formats.

In [ ]:
[plot_U]
parameter: data = path
# number of components to show
parameter: max_comp = -1
# whether or not to convert to correlation
parameter: to_cor = False
parameter: tol = "1E-6"
parameter: remove_label = False
parameter: name = ""
input: data
output: f'{cwd:a}/{_input:bn}{("_" + name.replace("$", "_")) if name != "" else ""}.pdf'
bash: expand = "${ }", stderr = f'{_output:n}.stderr', stdout = f'{_output:n}.stdout', container = container
    Rscript code/script/pecotmr_integration/mash_plot_prior.R \
        --data ${_input} \
        --max-comp ${max_comp} \
        --tol ${tol} \
        --name "${name}" \
        ${'--to-cor' if to_cor else ''} \
        ${'--remove-label' if remove_label else ''} \
        --output ${_output}